# Evaluate Performance Using Stacking Regressor of all previous models

Models to stack:
-   CatBoost
-   Ridge
-   SVR

In [ ]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import StackingRegressor
from sklearn.compose import TransformedTargetRegressor

ImportError: cannot import name 'TransformedTargetRegressor' from 'sklearn.preprocessing' (/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/sklearn/preprocessing/__init__.py)

In [3]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [4]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [5]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance


In [6]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

## **Preprocessing**

In [11]:
# Make pipeline for preprocessing steps:

preprocessor = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())    # impute and scale

In [18]:
# Going to train on whole feature set since model performance barely increased with feature selection.

estimators = [
    ('ridge', Ridge(alpha=0.16, solver='cholesky')),
    ('CatBoost', CatBoostRegressor(iterations=1322, depth=7, learning_rate=0.037, l2_leaf_reg=1.036, bagging_temperature=3.97, random_strength=0.241)),
    ('svr', SVR(C=382, epsilon=0.257, gamma=0.147))
]

reg = StackingRegressor(estimators=estimators, final_estimator=RandomForestRegressor(n_estimators=10))

model = make_pipeline(preprocessor, MultiOutputRegressor(reg))

In [20]:
# Fitting and evaluating model based on precomputed values in other notebooks.

model.fit(X_train, y_train)

0:	learn: 73.7140593	total: 2.74ms	remaining: 3.62s
1:	learn: 72.1124288	total: 7.33ms	remaining: 4.83s
2:	learn: 70.6374225	total: 9.34ms	remaining: 4.1s
3:	learn: 69.2863796	total: 11.1ms	remaining: 3.65s
4:	learn: 67.9454984	total: 12.7ms	remaining: 3.34s
5:	learn: 66.7254100	total: 14.2ms	remaining: 3.12s
6:	learn: 65.5387407	total: 15.9ms	remaining: 2.98s
7:	learn: 64.3566616	total: 17.7ms	remaining: 2.9s
8:	learn: 63.2517581	total: 19.6ms	remaining: 2.85s
9:	learn: 62.2182745	total: 21.1ms	remaining: 2.77s
10:	learn: 61.2124404	total: 22.5ms	remaining: 2.69s
11:	learn: 60.2436734	total: 23.9ms	remaining: 2.61s
12:	learn: 59.2430664	total: 25.4ms	remaining: 2.56s
13:	learn: 58.3697534	total: 27ms	remaining: 2.52s
14:	learn: 57.4717585	total: 28.7ms	remaining: 2.5s
15:	learn: 56.5958348	total: 30.3ms	remaining: 2.47s
16:	learn: 55.7906723	total: 31.7ms	remaining: 2.43s
17:	learn: 55.0959891	total: 33.1ms	remaining: 2.4s
18:	learn: 54.3558691	total: 35ms	remaining: 2.4s
19:	learn: 5

Pipeline(steps=[('pipeline',
                 Pipeline(steps=[('simpleimputer',
                                  SimpleImputer(strategy='median')),
                                 ('standardscaler', StandardScaler())])),
                ('multioutputregressor',
                 MultiOutputRegressor(estimator=StackingRegressor(estimators=[('ridge',
                                                                               Ridge(alpha=0.16,
                                                                                     solver='cholesky')),
                                                                              ('CatBoost',
                                                                               CatBoostRegressor(bagging_temperature=3.97, depth=7, iterations=1322, l2_leaf_reg=1.036, learning_rate=0.037, loss_function='RMSE', random_strength=0.241)),
                                                                              ('svr',
                                                                               SVR(C=382,
                                                                                   epsilon=0.257,
                                                                                   gamma=0.147))],
                                                                  final_estimator=RandomForestRegressor(n_estimators=10))))])

In [21]:
model.score(X_test, y_test)

0.7443831687514312

In [22]:
# Now try by transforming the target and iterative imputing

iterative_preprocess = make_pipeline(
    IterativeImputer((RandomForestRegressor(n_estimators=100, n_jobs=-1))),
    StandardScaler()
)

tr = TransformedTargetRegressor(reg, func=np.log1p, inverse_func=np.expm1)
trans_model = make_pipeline(iterative_preprocess, MultiOutputRegressor(tr))

trans_model.fit(X_train, y_train)

0:	learn: 0.8302230	total: 2.05ms	remaining: 2.7s
1:	learn: 0.8115883	total: 3.6ms	remaining: 2.38s
2:	learn: 0.7947300	total: 5.21ms	remaining: 2.29s
3:	learn: 0.7776820	total: 6.79ms	remaining: 2.24s
4:	learn: 0.7616220	total: 8.32ms	remaining: 2.19s
5:	learn: 0.7461440	total: 9.83ms	remaining: 2.16s
6:	learn: 0.7317695	total: 11.3ms	remaining: 2.12s
7:	learn: 0.7176873	total: 12.7ms	remaining: 2.09s
8:	learn: 0.7035626	total: 14.3ms	remaining: 2.08s
9:	learn: 0.6907099	total: 15.8ms	remaining: 2.07s
10:	learn: 0.6787276	total: 17.4ms	remaining: 2.07s
11:	learn: 0.6655171	total: 19ms	remaining: 2.07s
12:	learn: 0.6543017	total: 20.4ms	remaining: 2.05s
13:	learn: 0.6435338	total: 21.8ms	remaining: 2.04s
14:	learn: 0.6329177	total: 23.5ms	remaining: 2.04s
15:	learn: 0.6236545	total: 24.9ms	remaining: 2.03s
16:	learn: 0.6137579	total: 26.4ms	remaining: 2.03s
17:	learn: 0.6051988	total: 27.8ms	remaining: 2.01s
18:	learn: 0.5963917	total: 29.3ms	remaining: 2.01s
19:	learn: 0.5876842	total

Pipeline(steps=[('pipeline',
                 Pipeline(steps=[('iterativeimputer',
                                  IterativeImputer(estimator=RandomForestRegressor(n_jobs=-1))),
                                 ('standardscaler', StandardScaler())])),
                ('multioutputregressor',
                 MultiOutputRegressor(estimator=TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                                                           inverse_func=<ufunc 'expm1'>,
                                                                           regressor=StackingRegressor(estimators=[('ridge',
                                                                                                                    Ridge(alpha=0.16,
                                                                                                                          solver='cholesky')),
                                                                                                                   ('CatBoost',
                                                                                                                    CatBoostRegressor(bagging_temperature=3.97, depth=7, iterations=1322, l2_leaf_reg=1.036, learning_rate=0.037, loss_function='RMSE', random_strength=0.241)),
                                                                                                                   ('svr',
                                                                                                                    SVR(C=382,
                                                                                                                        epsilon=0.257,
                                                                                                                        gamma=0.147))],
                                                                                                       final_estimator=RandomForestRegressor(n_estimators=10)))))])

In [23]:
trans_model.score(X_test, y_test)

0.7131029164039164

Transforming the target and performing Iterative Imputation actually performed worse.